In [ ]:
import pandas as pd

# Read Reddit Data
reddit_comments = pd.read_csv("redditcomments.csv")
reddit_comments = reddit_comments.rename(columns={"cleaned_text": "processed_text", "comment": "text"})
yt_comments = pd.read_excel("method-1-youtube/scripts/youtube_comments.xlsx", sheet_name="youtube_base")

df = pd.concat([reddit_comments, yt_comments])
df

In [ ]:
text_col = "processed_text"

print("Using text column:", text_col)
print("Sample text:", df[text_col].dropna().astype(str).head(3).tolist())


In [ ]:

# --- Term-Frequency Language Model (Top 3 words) ---
import re
from sklearn.feature_extraction.text import CountVectorizer

def simple_clean(s: str) -> str:
    s = s.lower()
    s = re.sub(r"http\S+|www\S+", " ", s)      # remove urls
    s = re.sub(r"[^a-z0-9\s]", " ", s)         # keep alphanumerics
    s = re.sub(r"\s+", " ", s).strip()
    return s

texts = df[text_col].dropna().astype(str).apply(simple_clean).tolist()

# basic stopwords list (can be extended)
_stopwords_blob = (
    "a an the and or but if while to from in on at for by of is are was were be been being "
    "it this that these those i you he she they we us our your their my me his her its "
    "with as not no yes do does did have has had will would can could should may might "
    "about into over under out up down than then so such very just also more most other"
)
stopwords = set(_stopwords_blob.split())

def stop_filter(token):
    return token not in stopwords and len(token) > 2 and not token.isdigit()

# Vectorize
cv = CountVectorizer(token_pattern=r"(?u)\b\w+\b")
X = cv.fit_transform(texts)
vocab = cv.get_feature_names_out()

# Sum counts and pick top words (excluding stopwords)
import numpy as np
counts = np.asarray(X.sum(axis=0)).ravel()
freqs = sorted([(vocab[i], int(counts[i])) for i in range(len(vocab))], key=lambda x: -x[1])

top_words = []
for w, c in freqs:
    if stop_filter(w):
        top_words.append((w, c))
    if len(top_words) >= 3:
        break

print("Top 3 words (word, count):", top_words)


In [ ]:

# --- Train static Word2Vec and find most similar words for the top 3 ---
from gensim.models import Word2Vec

# Tokenize for Word2Vec
tokenized = [t.split() for t in texts]

# Small, fast Word2Vec just for demo; adjust as needed
w2v = Word2Vec(sentences=tokenized, vector_size=100, window=5, min_count=1, workers=2, sg=1, epochs=20)

neighbors = {}
for w, _ in top_words:
    if w in w2v.wv.key_to_index:
        neighbors[w] = w2v.wv.most_similar(w, topn=10)
    else:
        neighbors[w] = []

for w, sims in neighbors.items():
    print(f"\nMost similar to '{w}':")
    for term, score in sims:
        print(f"  {term:20s}  {score:.3f}")


In [ ]:

# --- PCA of the top words + their neighbors ---
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

plot_words = set([w for w,_ in top_words])
for _, sims in neighbors.items():
    for term,_ in sims[:8]:  # take first 8 neighbors per top word to avoid clutter
        plot_words.add(term)

# collect vectors
labels = []
vecs = []
for w in plot_words:
    if w in w2v.wv.key_to_index:
        labels.append(w)
        vecs.append(w2v.wv[w])
vecs = np.vstack(vecs)

# PCA to 2D
pca = PCA(n_components=2, random_state=0)
pts = pca.fit_transform(vecs)

# plot (one plot, no specific colors)
plt.figure(figsize=(7,6))
plt.scatter(pts[:,0], pts[:,1])
for i, label in enumerate(labels):
    plt.annotate(label, (pts[i,0], pts[i,1]))
plt.title("PCA of Static Embeddings: Top Words + Neighbors")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


In [ ]:
# --- Hugging Face BERT/RoBERTa Sentiment ---
from transformers import pipeline
import pandas as pd

# Choose one of these models:
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"   # better for social media / mixed lang
# MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"  # lightweight English-only

BATCH_SIZE = 64
MAX_ROWS = 1000

subset = df[text_col].dropna().astype(str).head(MAX_ROWS).tolist()

# Build pipeline (device=-1 → CPU, change to 0 if you have GPU + CUDA)
clf = pipeline("sentiment-analysis", model=MODEL_NAME, device=-1, truncation=True)
preds = clf(subset, batch_size=BATCH_SIZE)

# Convert to DataFrame
out = pd.DataFrame(preds)
out["text"] = subset

display(out.head())

# Label distribution
print("Sentiment label distribution:\n", out["label"].value_counts(dropna=False))


In [ ]:

# --- Hugging Face BERT-based Sentiment ---
# We'll try a compact model. If offline or uncached, we fall back to VADER.

from typing import List
BATCH_SIZE = 64
MAX_ROWS = 1000  # adjust as needed

subset = df[text_col].dropna().astype(str).head(MAX_ROWS).tolist()

preds = None
used_model = None
error_msg = None

try:
    from transformers import pipeline
    used_model = "cardiffnlp/twitter-roberta-base-sentiment-latest"
    clf = pipeline("sentiment-analysis", model=used_model, truncation=True)
    preds = clf(subset, batch_size=BATCH_SIZE)
except Exception as e:
    error_msg = str(e)
    preds = None

import pandas as pd

if preds is None:
    print("⚠️ Could not run the Hugging Face model (likely offline or missing cache).")
    if error_msg:
        print("Reason:", error_msg[:300], "...")
    print("Falling back to VADER (lexicon-based) so you still get a sentiment view.")
    try:
        from nltk.sentiment import SentimentIntensityAnalyzer
        import nltk
        try:
            nltk.data.find('sentiment/vader_lexicon.zip')
        except LookupError:
            nltk.download('vader_lexicon')
        sia = SentimentIntensityAnalyzer()
        vader_scores = [sia.polarity_scores(t) for t in subset]
        out = pd.DataFrame(vader_scores)
        out["text"] = subset
        display(out.head())
    except Exception as e2:
        print("VADER also failed:", e2)
else:
    out = pd.DataFrame(preds)
    out["text"] = subset
    display(out.head())

# Summary
if 'label' in out.columns:
    print("Sentiment label distribution:\n", out['label'].value_counts(dropna=False))
elif 'compound' in out.columns:
    print("VADER sentiment summary (compound scores):")
    print(out["compound"].describe())
